# Does Pre-Treatment Morphology Predict Eventual Fate? (vem 1uM, 48hrs Pre-Treatment)

Frame 1 (2 days before 1uM vemurafenib is added) cells don't have a
single fate label: `cloneClassification` only classifies cells present
at day 0 (frame 3), and by then a frame-1 lineage can have already split
into several day-0 descendants with *different* fates. Each frame-1 cell
instead gets a **mixture**: the fraction of its day-0 descendants that
end up dying, persisting, as a slow proliferator, or as a fast
proliferator (`lineage_fate.py`).

**Question**: does a cell's shape *before* treatment predict that
mixture? For each cell, this notebook predicts its fate mixture from the
mixtures of its k nearest neighbors *in real morphological distance*
(CAJAL's Gromov-Wasserstein distance matrix, not a UMAP/PHATE
projection, which distorts distances) and tests whether that prediction
is better than chance via permutation: repeatedly reassign which
mixture belongs to which cell at random, and see how often a random
reassignment predicts as well as the real one does.

Requires `main.py` to have already been run (for `selected_metadata.csv`
and `{cell,nuc}_qgw_075P_dists.csv` under `output/cajal_output/`), and
`config.RAW_DRUG_DIR` / `LINEAGE_MAT_PATH` / `CLONE_KMEANS_MAT_PATH` to
point at the real raw data.


In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/02_vem_48hr_pretreatment/notebooks/), so the analysis dir - and
# config.py - is one level up, and the repo root - where shared/ lives -
# is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR are relative to the analysis
# directory, not this notebook's own cwd - re-anchor them to an absolute
# path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.cajal.embed import load_distance_matrix
from shared.load_data import match_labels_to_metadata
from shared.morphology_predicts_fate import knn_prediction_permutation_test, plot_permutation_test
import lineage_fate


## Method

For each modality (cell, nucleus):

1. **Load the real morphological distance matrix** (`load_distance_matrix`)
   and this analysis's own selected-cell metadata, and match the two
   (`match_labels_to_metadata`) - no embedding involved, the distance
   matrix itself is the morphology representation.
2. **Resolve each frame-1 lineage's fate mixture** (`lineage_fate.py`,
   same as before): `pct_dying`/`pct_persisting`/`pct_slow_prolif`/`pct_fast_prolif`
   per lineage, merged in by `lin_id`. A cell whose lineage dies out
   before day 0 has no mixture to predict or predict from, so it's
   dropped from both the metadata AND the distance matrix (dropping a
   row from one without the other would misalign every remaining pair).
3. **Predict each cell's mixture from its k nearest morphological
   neighbors**' own (real) mixtures, and test significance by comparing
   that prediction's error against the same prediction made after
   randomly reassigning mixtures to cells, many times
   (`knn_prediction_permutation_test` - see its docstring for the exact
   method).
4. **Plot**: one figure per modality - the permutation null
   distribution with the real (observed) prediction error marked
   (`plot_permutation_test`).


In [ ]:
CATEGORY_NAMES = lineage_fate.SUBCLASSES
PCT_COLS = [f'pct_{c}' for c in CATEGORY_NAMES]

PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '01_morphology_predicts_fate')
SVG_DIR = os.path.join(PLOTS_DIR, 'svgs')
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}
os.makedirs(SVG_DIR, exist_ok=True)

# number of nearest morphological neighbors averaged for each cell's
# prediction, and number of random label reassignments making up the
# permutation test's null distribution
K_NEIGHBORS     = 10
N_PERMUTATIONS  = 10_000
SEED            = 42


def process_modality(modality, fate_pct):
    '''
    Load this modality's real distance matrix and metadata, attach each
    cell's fate mixture, run the KNN prediction + permutation test, and
    plot the result.

    Inputs:
        modality: 'cell' or 'nuc'
        fate_pct: from lineage_fate.compute_frame1_fate_percentages
    Outputs:
        observed_error, null_errors, p_value: see knn_prediction_permutation_test
    '''
    dist_csv = os.path.join(config.CAJAL_OUTPUT_DIR, f'{modality}_qgw_{config.N_POINTS:03d}P_dists.csv')
    labels, dist_matrix = load_distance_matrix(dist_csv)

    selected = pd.read_csv(os.path.join(config.CAJAL_OUTPUT_DIR, 'selected_metadata.csv'))
    df = match_labels_to_metadata(labels, selected)
    df = df.merge(fate_pct, on='lin_id', how='left')

    keep = df[PCT_COLS[0]].notna().to_numpy()
    n_dropped = (~keep).sum()
    if n_dropped > 0:
        print(f"  Dropping {n_dropped}/{len(df)} frame-1 cell(s) with no day-0 descendants in "
              f"lineage_frame3_to_frame1.mat (lineage died before day 0)")
    df = df[keep].reset_index(drop=True)
    dist_matrix = dist_matrix[keep][:, keep]  # same rows dropped on both axes, so pairs stay aligned

    mixture = df[PCT_COLS].to_numpy()
    print(f"[{modality}] {len(df)} frame-1 cells with a resolved fate mixture")

    observed_error, null_errors, p_value, _ = knn_prediction_permutation_test(
        dist_matrix, mixture, k=K_NEIGHBORS, n_permutations=N_PERMUTATIONS, seed=SEED,
    )
    print(f"  observed mean prediction error = {observed_error:.4f} "
          f"(null mean = {null_errors.mean():.4f}, p = {p_value:.4f})")

    fig = plot_permutation_test(null_errors, observed_error, p_value, title=f'{MODALITY_NAMES[modality]} morphology')
    fig.savefig(os.path.join(PLOTS_DIR, f'{modality}.pdf'), bbox_inches='tight')
    fig.savefig(os.path.join(SVG_DIR, f'{modality}.svg'), bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved to {PLOTS_DIR}")

    return observed_error, null_errors, p_value


raw_to_global = lineage_fate.build_raw_to_global_lin_id(config.RAW_DRUG_DIR)
fate_pct = lineage_fate.compute_frame1_fate_percentages(
    config.LINEAGE_MAT_PATH, config.CLONE_KMEANS_MAT_PATH, raw_to_global,
)
print(f"{len(fate_pct)} frame-1 lineages have at least one day-0 descendant")


## Run: cell morphology

In [ ]:
cell_error, cell_null_errors, cell_p_value = process_modality('cell', fate_pct)


## Run: nuclear morphology

In [ ]:
nuc_error, nuc_null_errors, nuc_p_value = process_modality('nuc', fate_pct)
